# Data_Preprocessing and Featurization

This pipeline ingests the raw Amazon ESCI dataset and transforms it into a production-ready format. It handles locale filtering (US-only), heuristic cleaning (removing non-English queries), and feature engineering (synthesizing product text). Crucially, it enforces a strict Train/Test split to prevent data leakage during evaluation.

In [1]:
from pathlib import Path
import sys, yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR)) 

from esci.data import (
    filter_queries_with_E,
    remove_train_test_overlap,
    add_product_text,        
    add_grades_and_pair_view,
    sample_dataset,
)

### Load config + paths

In [2]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "esci.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

examples_path = Path(cfg["paths"]["raw_examples"])
products_path = Path(cfg["paths"]["raw_products"])

processed_dir = Path(cfg["paths"]["processed_dir"])
processed_dir.mkdir(parents=True, exist_ok=True)

out_path = processed_dir / "pair_df.parquet"

### Load raw data

In [3]:
# Using PROJECT_ROOT
df_examples = pd.read_parquet(PROJECT_ROOT / examples_path)
df_products = pd.read_parquet(PROJECT_ROOT / products_path)

# Merge
df = pd.merge(
    df_examples, 
    df_products, 
    how='left', 
    left_on=['product_locale', 'product_id'], 
    right_on=['product_locale', 'product_id']
)

### Data Loading & Filtering

In [4]:
# We strictly filter for 'us' (United States) and 'small_version'.
# Multi-lingual retrieval requires different tokenizers; 
# mixing locales introduces noise that degrades English-centric models like BGE.

print(f"Total rows before filter: {len(df)}")
df = df[(df["product_locale"] == "us") & (df["small_version"] == 1)].copy() 
print(f"Rows after 'us' filter: {len(df)}")

Total rows before filter: 2621288
Rows after 'us' filter: 601354


### Noise Reduction

In [5]:
# We remove queries containing non-ASCII characters. 
# BGE-Base-English is not optimized for mixed-script queries (e.g., Korean/Chinese),
# which are present even in the 'us' locale subset.

print("Removing non-English queries...")
original_count = len(df)
df = df[df["query"].apply(lambda x: str(x).isascii())]
print(f"Dropped {original_count - len(df)} non-English rows.")
print(f"Final Count: {len(df)}")

Removing non-English queries...
Dropped 3726 non-English rows.
Final Count: 597628


In [6]:
df.isna().sum()

example_id                   0
query                        0
query_id                     0
product_id                   0
product_locale               0
esci_label                   0
small_version                0
large_version                0
split                        0
product_title                0
product_description     298458
product_bullet_point     69904
product_brand            29538
product_color           184068
dtype: int64

- Based on these results, we will build our "Product_Text" mainly with product_title, product_brand and product_bullet_point.
- Using product_description and product_color will introduce more harm (noise) than any good.

In [7]:
df

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
16,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","<b>About The Ram-Pro All Purpose Utility 10"" A...",✓ The Ram-Pro Ten Inch ready to install Air Ti...,RamPro,10 Inch
17,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,Please check your existing tire Sidewall for t...,MaxAuto,None
18,18,!awnmower tires without rims,1,B082K7V2GZ,us,I,1,1,train,NEIKO 20601A 14.5 inch Steel Tire Spoon Lever ...,None,[QUALITY]: Hardened Steel-Iron construction wi...,Neiko,None
19,19,!awnmower tires without rims,1,B07P4CF3DP,us,S,1,1,train,2PK 13x5.00-6 13x5.00x6 13x5x6 13x5-6 2PLY Tur...,"Tire Size: 13 x 5.00 - 6 Axle: 3/4"" inside dia...",None,Russo,None
20,20,!awnmower tires without rims,1,B07C1WZG12,us,E,1,1,train,(Set of 2) 15x6.00-6 Husqvarna/Poulan Tire Whe...,No fuss. Just take off your old assembly and r...,Tire size:15x6.00-6 Ply: 4 Tubeless\n6x4.5 Whe...,Antego Tire & Wheel,Husqvarna Silver
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260445,2260445,zxi 900,115921,B01BFSRAEY,us,C,1,1,train,Kawasaki Keihin CDk-2 Triple Carb Carburetor R...,NEW KEIHIN CDK-2 TRIPLE CARB REBUILD KIT Inclu...,None,TITAN 757 PERFORMANCE,None
2260446,2260446,zxi 900,115921,B01DSYJFZ0,us,C,1,1,train,JSP Manufacturing White Drain Plug Replacement...,<b>JSP Manufacturing® Drain Plug & O-Ring Only...,Aftermarket Kawasaki White Drain Plug\nReplace...,JSP Manufacturing,White
2260447,2260447,zxi 900,115921,B01F0D80US,us,C,1,1,train,3 Pack JSP Brand DUCT FLAME ARRESTOR Fits KAWA...,<b>JSP Manufacturing® Intake Duct Flame Arrest...,New Aftermarket Kawasaki Intake Duct Flame Arr...,JSP Manufacturing,None
2260448,2260448,zxi 900,115921,B01FR8TDJ8,us,C,1,1,train,"STORAGE COVER for KAWASAKI JET SKI 900, 1100 Z...",STORAGE JET SKI COVER. DESIGNED FOR MOORING OR...,Economy Storage Personal Watercraft - JetSki C...,STOPBYUS,silver


In [8]:
# Label Distribution
df['esci_label'].value_counts()

esci_label
E    260221
S    210028
I    100328
C     27051
Name: count, dtype: int64

In [9]:
# This function is only activated when debog mode is "True"
df = sample_dataset(df, cfg)

In [10]:
# We keep only queries that contain at least one "E" product

df = filter_queries_with_E(df)

Before filtering: 29,687 queries total
After filtering (require E): 29,686 queries total
product_locale
us    29686
Name: queries_with_E, dtype: int64


### Leakage Prevention

In [11]:
# Standard random splitting is insufficient because the same 'query_id' 
# often appears multiple times with different products.
# We must ensure that a query seen in Train is NEVER seen in Test.
# we found an overlap when using "large_version" dataset.

df = remove_train_test_overlap(df)

 Overlapping queries between train and test: 0
After removing overlap: train queries = 20,778, test queries = 8,908


In [12]:
# This function builds product_text_dense / product_text_sparse / query_sparse

df = add_product_text(df)

Generating product text representations...


In [13]:
# Here we create pair-level dataframe with grades

df = add_grades_and_pair_view(df)

Pair-level DataFrame shape: (597613, 10)


In [14]:
# We save the final processed dataset

df.to_parquet(out_path)
print(f"Saved pair_df to: {out_path}")
print("pair_df columns:", list(df.columns))

Saved pair_df to: ..\data\processed\pair_df.parquet
pair_df columns: ['example_id', 'query_id', 'product_id', 'product_locale', 'query', 'esci_label', 'grade', 'split', 'product_title', 'product_text_dense']


In [15]:
df

,example_id,query_id,product_id,product_locale,query,esci_label,grade,split,product_title,product_text_dense
0,16,1,B075SCHMPY,us,!awnmower tires without rims,I,0.00,train,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","RamPro 10"" All Purpose Utility Air Tires/Wheel..."
1,17,1,B08L3B9B9P,us,!awnmower tires without rims,E,1.00,train,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...
2,18,1,B082K7V2GZ,us,!awnmower tires without rims,I,0.00,train,NEIKO 20601A 14.5 inch Steel Tire Spoon Lever ...,NEIKO 20601A 14.5 inch Steel Tire Spoon Lever ...
3,19,1,B07P4CF3DP,us,!awnmower tires without rims,S,0.10,train,2PK 13x5.00-6 13x5.00x6 13x5x6 13x5-6 2PLY Tur...,2PK 13x5.00-6 13x5.00x6 13x5x6 13x5-6 2PLY Tur...
4,20,1,B07C1WZG12,us,!awnmower tires without rims,E,1.00,train,(Set of 2) 15x6.00-6 Husqvarna/Poulan Tire Whe...,(Set of 2) 15x6.00-6 Husqvarna/Poulan Tire Whe...
...,...,...,...,...,...,...,...,...,...,...
597608,2260445,115921,B01BFSRAEY,us,zxi 900,C,0.01,train,Kawasaki Keihin CDk-2 Triple Carb Carburetor R...,Kawasaki Keihin CDk-2 Triple Carb Carburetor R...
597609,2260446,115921,B01DSYJFZ0,us,zxi 900,C,0.01,train,JSP Manufacturing White Drain Plug Replacement...,JSP Manufacturing White Drain Plug Replacement...
597610,2260447,115921,B01F0D80US,us,zxi 900,C,0.01,train,3 Pack JSP Brand DUCT FLAME ARRESTOR Fits KAWA...,3 Pack JSP Brand DUCT FLAME ARRESTOR Fits KAWA...
597611,2260448,115921,B01FR8TDJ8,us,zxi 900,C,0.01,train,"STORAGE COVER for KAWASAKI JET SKI 900, 1100 Z...","STORAGE COVER for KAWASAKI JET SKI 900, 1100 Z..."


**"product_text_dense"** is the Text used for Dense and SPLADE Retrieval because these two models excel at semantic understanding. This will allow them to capture synonyms and context from the entire product profile.
For BM25 Retrieval, we will be focusing only on "product_title" because BM25 relies strictly on exact keyword matching. Therefore, we wanted to prevent irrelevant matches caused by "noisy" or repetitive vocabulary in long descriptions.

In [16]:
# Here we want to have an idea about the token distribution in each "product_text_dense"

model_name = cfg["biencoder_model"]
tokenizer = AutoTokenizer.from_pretrained(model_name)

def get_token_length(text):
    # Truncation=False ensures we count EVERYTHING
    # add_special_tokens=True adds [CLS] and [SEP]
    tokens = tokenizer(str(text), truncation=False, padding=False, add_special_tokens=True)
    return len(tokens["input_ids"])

print("Calculating token lengths... this might take a minute...")
lengths = df["product_text_dense"].apply(get_token_length)

# Print Statistics
print("\n--- Token Length Statistics ---")
print(f"Mean Length:   {lengths.mean():.1f}")
print(f"Median Length: {lengths.median():.1f}")
print(f"Min Length:    {lengths.min()}")
print(f"Max Length:    {lengths.max()}")

# Critical Percentiles
print("\n--- Percentiles (How much are you truncating?) ---")
for p in [50, 75, 90, 95, 99]:
    val = np.percentile(lengths, p)
    status = "Safe" if val <= 256 else "Truncated"
    print(f"{p}th Percentile: {val:.0f} tokens -> {status}")

Calculating token lengths... this might take a minute...


Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors



--- Token Length Statistics ---
Mean Length:   170.9
Median Length: 149.0
Min Length:    2
Max Length:    1602

--- Percentiles (How much are you truncating?) ---
50th Percentile: 149 tokens -> Safe
75th Percentile: 253 tokens -> Safe
90th Percentile: 342 tokens -> Truncated
95th Percentile: 402 tokens -> Truncated
99th Percentile: 510 tokens -> Truncated


Between what we want (512 tokens) and what our hardware can handle (up to 256 tokens). Our choice was to define max_seq_length = 256, which is not bad giving the fact that more than 75% of the product texts are safe.